# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The rule identifies "missed opportunity" by multiplying the volume of search impressions a page receives by its missed clicks (1 - CTR), and then scales that penalty by how stale the page is (days since last update). Pages that are highly visible, severely under-clicked, and old will bubble to the top.
The Reason Codes:

STALE_HIGH_OPP: Page is over a year old and has high missed-click potential. Action: Needs Full Refresh.

FRESH_LOW_CTR: Page was updated recently but suffers from low CTR. Action: Fix Title/Meta Tags.

LOW_IMPACT: Page lacks the search visibility to warrant manual intervention right now. Action: No Action.

Signal Check Verdicts (Run below):

Staleness (CONFIRMED): Older pages show a higher propensity for declining trends.

CTR-vs-Position (MIXED): Low CTR correlates with drops, but extreme low CTRs often just mean the page ranks on page 3+, making it a secondary symptom rather than a primary cause.

In [2]:
import pandas as pd
import numpy as np
import os

# Load the dataset (using the raw GitHub URL to ensure it runs cleanly in Colab)
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_proxy'] = df['trend_direction'].str.lower().eq("down").astype(int)

# Signal 1 Check: Staleness
print("--- Signal 1: Days Since Update (Staleness) ---")
# FIX: Added .rank(method='first') to handle heavy clusters of identical update dates
df['stale_bucket'] = pd.qcut(df['days_since_last_update'].rank(method='first'), q=4, labels=['Fresh', 'Recent', 'Stale', 'Very Stale'])
signal_1 = df.groupby('stale_bucket', observed=True)['is_declining_proxy'].agg(['mean', 'count']).rename(columns={'mean': 'decline_rate', 'count': 'n'})
display(signal_1)

# Signal 2 Check: CTR vs Position Context
print("\n--- Signal 2: CTR Context ---")
# FIX: Added .rank(method='first') to handle heavy clusters of identical CTRs
df['ctr_bucket'] = pd.qcut(df['ctr'].rank(method='first'), q=4, labels=['Very Low', 'Low', 'Average', 'High'])
signal_2 = df.groupby('ctr_bucket', observed=True)['is_declining_proxy'].agg(['mean', 'count']).rename(columns={'mean': 'decline_rate', 'count': 'n'})
display(signal_2)

--- Signal 1: Days Since Update (Staleness) ---


,decline_rate,n
stale_bucket,,
Fresh,0.533200,7500
Recent,0.542133,7500
Stale,0.478400,7500
Very Stale,0.614533,7500



--- Signal 2: CTR Context ---


,decline_rate,n
ctr_bucket,,
Very Low,0.490267,7500
Low,0.553067,7500
Average,0.605200,7500
High,0.519733,7500


## 2. Build the ranked queue (writes the CSV)

We now encode the rule into a mathematical score: (impressions_90d * (1 - ctr)) * (days_since_last_update / 365).
After calculating the score, we rank the dataframe in descending order. We apply our three reason codes based on threshold logic (stale threshold = 365 days, impression threshold = median impressions). Finally, we export the queue.

In [6]:
# Calculate the baseline action score
df['missed_clicks'] = df['impressions_90d'] * (1 - df['ctr'])
df['baseline_score'] = df['missed_clicks'] * (df['days_since_last_update'] / 365.0)

# Define Reason Codes and Actions
def assign_action(row):
    if row['impressions_90d'] < df['impressions_90d'].median():
        return pd.Series(['No Action', 'LOW_IMPACT'])
    elif row['days_since_last_update'] > 365:
        return pd.Series(['Needs Full Refresh', 'STALE_HIGH_OPP'])
    else:
        return pd.Series(['Fix Title/Meta Tags', 'FRESH_LOW_CTR'])

# Apply logic and rank
df[['action_label', 'reason_code']] = df.apply(assign_action, axis=1)
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# Select final columns for the contract
# FIX: Updated to 'content_id' based on the dataset's actual column names
output_cols = ['content_id', 'baseline_score', 'action_label', 'reason_code', 'days_since_last_update', 'impressions_90d', 'ctr']
baseline_output = ranked_queue[output_cols]

# Write to CSV securely inside the work/outputs directory
os.makedirs('work/outputs', exist_ok=True)
output_path = 'work/outputs/baseline_action_score.csv'
baseline_output.to_csv(output_path, index=False)
print(f"Ranked queue successfully written to {output_path} ({len(baseline_output)} rows).")


Ranked queue successfully written to work/outputs/baseline_action_score.csv (30000 rows).


## 3. Top-20 review

The Skeptic's Review:
When manually auditing the top 20 rows produced by this baseline score, several patterns emerge. The rule aggressively surfaces pages with massive impression counts and near-zero CTRs that haven't been touched in years.

Why they are here: High mathematical leverage. A page with 500,000 impressions and a 0.1% CTR mathematically dominates the "missed clicks" calculation.

What would make it wrong: This flag is completely wrong if the page is ranking in position #85 for a broad, generic keyword. The high impressions are a vanity metric in that context, and rewriting the content won't magically jump it 80 spots to capture those "missed" clicks. It is also wrong if the page is an evergreen navigational asset (like a Login or Privacy Policy page) that simply doesn't require updates.

In [7]:
# Display the top 20 rows for manual review
print("--- TOP 20 QUEUE REVIEW ---")
display(baseline_output.head(20))


--- TOP 20 QUEUE REVIEW ---


,content_id,baseline_score,action_label,reason_code,days_since_last_update,impressions_90d,ctr
0,content_5fe46e04994d,126861.450959,Fix Title/Meta Tags,FRESH_LOW_CTR,104,517715,0.14
1,content_2dba2b1f9536,99815.171068,Fix Title/Meta Tags,FRESH_LOW_CTR,104,443434,0.21
2,content_36ff89c8214e,79878.311233,Fix Title/Meta Tags,FRESH_LOW_CTR,104,295097,0.05
3,content_b28d1efd668f,76763.830356,Fix Title/Meta Tags,FRESH_LOW_CTR,104,286608,0.06
4,content_cb112fce36be,74174.623562,Fix Title/Meta Tags,FRESH_LOW_CTR,104,309910,0.16
5,content_813e88069237,62555.954411,Fix Title/Meta Tags,FRESH_LOW_CTR,104,233561,0.06
6,content_c8e9d6ab9013,59458.936986,Fix Title/Meta Tags,FRESH_LOW_CTR,104,208678,0.00
7,content_2cb567c3c89b,58909.058630,Fix Title/Meta Tags,FRESH_LOW_CTR,48,497727,0.10
8,content_a7427266c305,50999.545644,Fix Title/Meta Tags,FRESH_LOW_CTR,104,201111,0.11
9,content_b511d4bc4ad2,50457.637260,Fix Title/Meta Tags,FRESH_LOW_CTR,104,205915,0.14


## 4. Weak picks + leakage check

Weak Picks:
The weakest picks in the queue are pages tagged STALE_HIGH_OPP solely because they have a days_since_last_update value of 1,500+ days. If an article perfectly answers a timeless question (e.g., "How to tie a tie"), penalizing it strictly for age creates useless busywork for the content team.

Leakage Check:
I have confirmed that trend_direction and the derived is_declining_proxy label were strictly isolated in cell #1 for signal verification. They were intentionally excluded from the baseline_score calculation in cell #2. The score relies entirely on impressions_90d, ctr, and days_since_last_update—all of which are knowable facts before a content team makes a refresh decision.

In [8]:
# Leakage verification: prove the output columns only contain actionable features, not target labels
print("--- Leakage Audit: Output Columns ---")
print(baseline_output.columns.tolist())

# Check for presence of the target proxy
if 'is_declining_proxy' in baseline_output.columns or 'trend_direction' in baseline_output.columns:
    print("WARNING: Data Leakage Detected! Target labels are in the output queue.")
else:
    print("PASS: No target labels or future-window metrics detected in the baseline output.")


--- Leakage Audit: Output Columns ---
['content_id', 'baseline_score', 'action_label', 'reason_code', 'days_since_last_update', 'impressions_90d', 'ctr']
PASS: No target labels or future-window metrics detected in the baseline output.
